# TRIBE Validation Analysis (v4) — Deep Diagnostics

**Goal:** explain *why* TRIBE features don't yield predictive lift over baseline despite showing significant univariate ROI correlations.

**Tests:**
1. **ROI-148 features** (Destrieux mean per region) instead of raw 20484. The single most likely fix.
2. **fusion_hidden** as alternative TRIBE feature (1152 → PCA-30).
3. **TRIBE-only** classification (no baseline). What does TRIBE achieve alone?
4. **Partial correlations** ROI vs log_views | log_followers. Does TRIBE add orthogonal signal to reach?
5. **TRIBE residual** analysis. Predict log_views from baseline, check if TRIBE explains residuals.
6. **Bootstrap CI on delta**. Is -0.045 distinguishable from 0?
7. **Engagement outcome** (views/followers ratio) instead of raw views.

## 1. Setup and load (same as v3)

In [1]:
import os, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path('..')
CSV_PATH = PROJECT_ROOT / 'data' / 'videos.csv'
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'analysis_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
np.random.seed(42)
RANDOM_STATE = 42

meta = pd.read_csv(CSV_PATH)
available_npz = set(f.stem for f in FEATURES_DIR.glob('*.npz'))
meta['video_id'] = meta['video_id'].astype(str)
meta = meta[meta['video_id'].isin(available_npz)].copy().reset_index(drop=True)

fmri_means = []
fusion_means = []
for vid in meta['video_id']:
    npz = np.load(FEATURES_DIR / f'{vid}.npz', allow_pickle=True)
    fmri_means.append(npz['fmri_preds'].mean(axis=0))
    fusion = npz['fusion_hidden']
    fusion_means.append(fusion.reshape(-1, fusion.shape[-1]).mean(axis=0))
fmri_means = np.stack(fmri_means)
fusion_means = np.stack(fusion_means)

meta['log_followers'] = np.log1p(pd.to_numeric(meta['creator_followers_at_post'], errors='coerce').fillna(0))
meta['log_views'] = np.log1p(pd.to_numeric(meta['views_at_30d'], errors='coerce').fillna(0))
meta['duration_num'] = pd.to_numeric(meta['duration'], errors='coerce').fillna(0)
X_baseline = meta[['log_followers', 'duration_num']].values

binary_mask = meta['label'].isin(['viral', 'dud'])
y = (meta.loc[binary_mask, 'label'] == 'viral').astype(int).values
Xb = X_baseline[binary_mask.values]
Xt_fmri = fmri_means[binary_mask.values]
Xt_fusion = fusion_means[binary_mask.values]

print(f'Loaded {len(meta)} videos; binary task n={len(y)} (viral={y.sum()}, dud={(1-y).sum()})')
print(f'fmri_means: {fmri_means.shape}, fusion_means: {fusion_means.shape}')

Loaded 69 videos; binary task n=61 (viral=26, dud=35)
fmri_means: (69, 20484), fusion_means: (69, 1152)


## 2. Build ROI-148 features (Destrieux means) — Test 1

Replace the 20484 raw vertices with 148 anatomically meaningful ROIs. Each ROI is the mean activation across its constituent vertices.

In [2]:
from nilearn import datasets as nl_datasets
destrieux = nl_datasets.fetch_atlas_surf_destrieux()
labels = [l.decode() if isinstance(l, bytes) else l for l in destrieux['labels']]
left_annot = destrieux['map_left']
right_annot = destrieux['map_right']
annot = np.concatenate([left_annot, right_annot])
n_lab = len(labels)
annot_offset = annot.copy()
annot_offset[len(left_annot):] = annot[len(left_annot):] + n_lab
hemi_labels = ['L_'+l for l in labels] + ['R_'+l for l in labels]
n_rois = len(hemi_labels)

# Compute ROI means for ALL videos
roi_features = np.zeros((len(meta), n_rois))
for r in range(n_rois):
    mask = (annot_offset == r)
    if mask.sum() > 0:
        roi_features[:, r] = fmri_means[:, mask].mean(axis=1)

# Drop ROIs with zero variance (e.g., 'Unknown' label)
active_rois = roi_features.std(axis=0) > 1e-9
roi_features = roi_features[:, active_rois]
active_roi_names = [n for i, n in enumerate(hemi_labels) if active_rois[i]]
print(f'Active ROIs: {roi_features.shape[1]} (out of {n_rois})')

Xt_roi = roi_features[binary_mask.values]

[fetch_atlas_surf_destrieux] Dataset found in /Users/shloka/nilearn_data/destrieux_surface

Active ROIs: 150 (out of 152)


/var/folders/2y/fhw6y9y153z6y_j9bthwbkfm0000gn/T/ipykernel_42249/238052325.py:2: UserWarning: 
The following regions are present in the atlas look-up table,
but missing from the atlas image:

 index    name
     0 Unknown

  destrieux = nl_datasets.fetch_atlas_surf_destrieux()
/var/folders/2y/fhw6y9y153z6y_j9bthwbkfm0000gn/T/ipykernel_42249/238052325.py:2: UserWarning: 
The following regions are present in the atlas look-up table,
but missing from the atlas image:

 index    name
     0 Unknown

  destrieux = nl_datasets.fetch_atlas_surf_destrieux()


## 3. Helper: CV AUC for arbitrary feature combination

In [3]:
def cv_auc(features_list, y, n_splits=5, n_components=None, C=1.0, return_folds=False):
    """
    features_list: list of feature matrices (each (n, d_i)) to concatenate AFTER per-fold scaling.
    n_components: if int, applies PCA(n_components) to features that have d > 50 (i.e., TRIBE).
    Returns mean, std AUC across folds.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs = []
    for tr, te in skf.split(features_list[0], y):
        parts_tr, parts_te = [], []
        for X in features_list:
            sc = StandardScaler()
            X_tr = sc.fit_transform(X[tr])
            X_te = sc.transform(X[te])
            if n_components is not None and X.shape[1] > 50:
                k = min(n_components, X_tr.shape[0]-1, X_tr.shape[1])
                pca = PCA(n_components=k, random_state=RANDOM_STATE)
                X_tr = pca.fit_transform(X_tr)
                X_te = pca.transform(X_te)
            parts_tr.append(X_tr)
            parts_te.append(X_te)
        Xtr_full = np.hstack(parts_tr)
        Xte_full = np.hstack(parts_te)
        clf = LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE)
        clf.fit(Xtr_full, y[tr])
        aucs.append(roc_auc_score(y[te], clf.predict_proba(Xte_full)[:, 1]))
    if return_folds:
        return float(np.mean(aucs)), float(np.std(aucs)), aucs
    return float(np.mean(aucs)), float(np.std(aucs))

print('Helper defined')

Helper defined


## 4. Battery — TRIBE feature variants

In [4]:
results = {}

# Baseline alone
results['Baseline'] = cv_auc([Xb], y)

# TRIBE alone (each variant)
results['TRIBE fmri raw 20484 (alone)'] = cv_auc([Xt_fmri], y, n_components=30)
results['TRIBE ROI-148 (alone)'] = cv_auc([Xt_roi], y)
results['TRIBE fusion 1152 (alone)'] = cv_auc([Xt_fusion], y, n_components=30)

# Baseline + TRIBE
results['Baseline + TRIBE fmri PCA-30'] = cv_auc([Xb, Xt_fmri], y, n_components=30)
results['Baseline + TRIBE ROI-148'] = cv_auc([Xb, Xt_roi], y)
results['Baseline + TRIBE fusion PCA-30'] = cv_auc([Xb, Xt_fusion], y, n_components=30)

# Stronger regularization variant on ROI
results['Baseline + TRIBE ROI-148 (C=0.1)'] = cv_auc([Xb, Xt_roi], y, C=0.1)

print('=' * 70)
print(f'{"Model":<45} {"AUC":<8} {"std":<8}  delta')
print('=' * 70)
baseline_mean = results['Baseline'][0]
for name, (m, s) in results.items():
    delta = m - baseline_mean
    sign = '+' if delta >= 0 else ''
    print(f'{name:<45} {m:.3f}    {s:.3f}    {sign}{delta:.3f}')

Model                                         AUC      std       delta
Baseline                                      0.790    0.077    +0.000
TRIBE fmri raw 20484 (alone)                  0.670    0.232    -0.119
TRIBE ROI-148 (alone)                         0.677    0.269    -0.112
TRIBE fusion 1152 (alone)                     0.561    0.228    -0.229
Baseline + TRIBE fmri PCA-30                  0.670    0.229    -0.120
Baseline + TRIBE ROI-148                      0.729    0.186    -0.061
Baseline + TRIBE fusion PCA-30                0.547    0.222    -0.243
Baseline + TRIBE ROI-148 (C=0.1)              0.735    0.159    -0.054


## 5. Bootstrap CI on the best TRIBE delta — Test 6

Bootstrap resampling at the video level to put a 95% CI around the AUC delta. This tells us whether observed deltas are statistically distinguishable from 0.

In [5]:
def cv_auc_paired(Xb, Xt, y, n_components=None, C=1.0, n_splits=5):
    """Returns matched fold AUCs for baseline and baseline+TRIBE, same fold splits."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    auc_b, auc_t = [], []
    for tr, te in skf.split(Xb, y):
        sc_b = StandardScaler()
        Xb_tr = sc_b.fit_transform(Xb[tr])
        Xb_te = sc_b.transform(Xb[te])
        clf_b = LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE).fit(Xb_tr, y[tr])
        auc_b.append(roc_auc_score(y[te], clf_b.predict_proba(Xb_te)[:, 1]))

        sc_t = StandardScaler()
        Xt_tr = sc_t.fit_transform(Xt[tr])
        Xt_te = sc_t.transform(Xt[te])
        if n_components is not None and Xt.shape[1] > 50:
            k = min(n_components, Xt_tr.shape[0]-1, Xt_tr.shape[1])
            pca = PCA(n_components=k, random_state=RANDOM_STATE)
            Xt_tr = pca.fit_transform(Xt_tr)
            Xt_te = pca.transform(Xt_te)
        Xfull_tr = np.hstack([Xb_tr, Xt_tr])
        Xfull_te = np.hstack([Xb_te, Xt_te])
        clf_t = LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE).fit(Xfull_tr, y[tr])
        auc_t.append(roc_auc_score(y[te], clf_t.predict_proba(Xfull_te)[:, 1]))
    return np.array(auc_b), np.array(auc_t)

# Bootstrap on the BEST TRIBE variant (whichever had highest AUC)
best_name = max([k for k in results if 'TRIBE' in k and 'alone' not in k], key=lambda k: results[k][0])
print(f'Bootstrapping CI on: {best_name}\n')

# Determine features and PCA setting from name
if 'fmri' in best_name:
    Xt_best = Xt_fmri
    pca_setting = 30
elif 'ROI-148' in best_name:
    Xt_best = Xt_roi
    pca_setting = None
elif 'fusion' in best_name:
    Xt_best = Xt_fusion
    pca_setting = 30
C_setting = 0.1 if 'C=0.1' in best_name else 1.0

# Bootstrap the delta
n_boot = 500
boot_deltas = []
rng = np.random.RandomState(RANDOM_STATE)
for b in range(n_boot):
    idx = rng.choice(len(y), size=len(y), replace=True)
    if len(np.unique(y[idx])) < 2:
        continue
    try:
        auc_b, auc_t = cv_auc_paired(Xb[idx], Xt_best[idx], y[idx], n_components=pca_setting, C=C_setting)
        boot_deltas.append(auc_t.mean() - auc_b.mean())
    except Exception:
        continue

boot_deltas = np.array(boot_deltas)
ci_low, ci_high = np.percentile(boot_deltas, [2.5, 97.5])
print(f'Bootstrap delta: median = {np.median(boot_deltas):+.3f}')
print(f'95% CI: [{ci_low:+.3f}, {ci_high:+.3f}]')
print(f'Includes 0? {ci_low <= 0 <= ci_high}')
if ci_low > 0:
    print('=> TRIBE significantly improves over baseline.')
elif ci_high < 0:
    print('=> TRIBE significantly worse than baseline.')
else:
    print('=> Statistically indistinguishable from baseline at n=61.')

Bootstrapping CI on: Baseline + TRIBE ROI-148 (C=0.1)

Bootstrap delta: median = +0.018
95% CI: [-0.112, +0.150]
Includes 0? True
=> Statistically indistinguishable from baseline at n=61.


## 6. Partial correlations: do top ROIs survive controlling for log_followers? — Test 4

If high-follower creators systematically make content that activates left prefrontal regions, then ROI correlations are mediated by reach, not orthogonal signal.

**Partial correlation:** corr(ROI residual, log_views residual) where both are residualized against log_followers.

In [6]:
# Use all rows where log_views > 0
valid = (meta['log_views'] > 0).values
log_views = meta.loc[valid, 'log_views'].values
log_foll = meta.loc[valid, 'log_followers'].values.reshape(-1, 1)
roi_valid = roi_features[valid]

# Residualize log_views and each ROI against log_followers
lr_views = LinearRegression().fit(log_foll, log_views)
y_resid = log_views - lr_views.predict(log_foll)

rho_simple = np.zeros(roi_valid.shape[1])
rho_partial = np.zeros(roi_valid.shape[1])
p_simple = np.ones(roi_valid.shape[1])
p_partial = np.ones(roi_valid.shape[1])

for r in range(roi_valid.shape[1]):
    if roi_valid[:, r].std() > 1e-9:
        rho_simple[r], p_simple[r] = spearmanr(roi_valid[:, r], log_views)
        # Partial: residualize ROI against log_followers, then correlate residuals
        lr_roi = LinearRegression().fit(log_foll, roi_valid[:, r])
        roi_resid = roi_valid[:, r] - lr_roi.predict(log_foll)
        rho_partial[r], p_partial[r] = spearmanr(roi_resid, y_resid)

comp_df = pd.DataFrame({
    'roi': active_roi_names,
    'rho_simple': rho_simple,
    'p_simple': p_simple,
    'rho_partial': rho_partial,
    'p_partial': p_partial,
    'attenuation': np.abs(rho_simple) - np.abs(rho_partial),
})

# Sort by simple correlation magnitude, show top 10
comp_top = comp_df.reindex(comp_df['rho_simple'].abs().sort_values(ascending=False).index).head(10)
print('Top 10 ROIs (by simple correlation), with partial correlation comparison:\n')
print(comp_top.to_string(index=False))

n_significant_simple = (comp_df['p_simple'] < 0.01).sum()
n_significant_partial = (comp_df['p_partial'] < 0.01).sum()
print(f'\nROIs with p<0.01:')
print(f'  Simple correlation:  {n_significant_simple}/{len(comp_df)}')
print(f'  Partial correlation: {n_significant_partial}/{len(comp_df)}')
if n_significant_partial >= n_significant_simple * 0.5:
    print(f'  => Most ROI signal SURVIVES controlling for reach. TRIBE captures something orthogonal.')
elif n_significant_partial >= 5:
    print(f'  => SOME ROI signal survives. TRIBE captures a weaker orthogonal signal beyond reach.')
else:
    print(f'  => Most ROI signal DISAPPEARS when controlling for reach. The univariate correlations are mediated by log_followers.')

comp_df.to_csv(OUTPUT_DIR / 'partial_correlations.csv', index=False)
print(f'\nSaved {OUTPUT_DIR}/partial_correlations.csv')

Top 10 ROIs (by simple correlation), with partial correlation comparison:

                    roi  rho_simple  p_simple  rho_partial  p_partial  attenuation
    L_S_orbital_lateral    0.413043  0.000420     0.270150   0.024771     0.142894
          L_G_front_sup    0.412788  0.000424     0.193789   0.110603     0.218999
L_G_front_inf-Opercular    0.405627  0.000545     0.201790   0.096367     0.203836
 L_S_interm_prim-Jensen    0.403873  0.000579     0.192875   0.112326     0.210997
  L_G_front_inf-Orbital    0.403361  0.000589     0.245232   0.042259     0.158129
 L_G_front_inf-Triangul    0.399927  0.000663     0.214286   0.077049     0.185641
  R_G_front_inf-Orbital    0.370588  0.001721     0.166533   0.171430     0.204056
          R_G_front_sup    0.350895  0.003115     0.135440   0.267163     0.215455
    L_G_temporal_middle    0.341798  0.004048     0.065290   0.594030     0.276507
 L_Lat_Fis-ant-Horizont    0.337267  0.004598     0.244611   0.042800     0.092656

ROIs with p

## 7. TRIBE-residual analysis — Test 5

Predict log_views from baseline. Take residuals. Check if any TRIBE feature correlates with residuals. Isolates orthogonal TRIBE signal independent of any feature combination.

In [7]:
Xb_full = X_baseline[valid]
lr_b = LinearRegression().fit(Xb_full, log_views)
baseline_pred = lr_b.predict(Xb_full)
residuals = log_views - baseline_pred  # what baseline can't explain

print(f'Baseline R^2 on log_views: {lr_b.score(Xb_full, log_views):.3f}')
print(f'Residual std: {residuals.std():.3f}')

# Correlate every ROI with residuals
roi_resid_rho = np.zeros(roi_valid.shape[1])
roi_resid_p = np.ones(roi_valid.shape[1])
for r in range(roi_valid.shape[1]):
    if roi_valid[:, r].std() > 1e-9:
        roi_resid_rho[r], roi_resid_p[r] = spearmanr(roi_valid[:, r], residuals)

resid_df = pd.DataFrame({
    'roi': active_roi_names,
    'rho_vs_residual': roi_resid_rho,
    'p_vs_residual': roi_resid_p,
}).reindex(np.abs(roi_resid_rho).argsort()[::-1]).head(15)

print(f'\nTop 15 ROIs correlated with baseline-residual log_views:')
print(resid_df.to_string(index=False))

n_sig_resid = (roi_resid_p < 0.01).sum()
expected_by_chance = len(roi_resid_p) * 0.01
print(f'\nROIs with p<0.01 vs residuals: {n_sig_resid}')
print(f'Expected by chance: {expected_by_chance:.1f}')
if n_sig_resid > 3 * expected_by_chance:
    print('=> TRIBE explains baseline-orthogonal variance in log_views (genuine signal).')
else:
    print('=> TRIBE adds little orthogonal explanatory power to baseline.')

Baseline R^2 on log_views: 0.398
Residual std: 3.222

Top 15 ROIs correlated with baseline-residual log_views:
                       roi  rho_vs_residual  p_vs_residual
       L_S_orbital_lateral         0.282864       0.018521
     L_G_front_inf-Orbital         0.278115       0.020676
    L_Lat_Fis-ant-Horizont         0.245013       0.042449
    L_G_front_inf-Triangul         0.225649       0.062286
   L_G_front_inf-Opercular         0.217793       0.072221
             L_G_front_sup         0.216661       0.073753
    L_S_interm_prim-Jensen         0.212094       0.080194
               R_G_orbital         0.201717       0.096490
     R_G_front_inf-Orbital         0.196310       0.105953
               L_G_orbital         0.193789       0.110603
          L_G_front_middle         0.192693       0.112674
R_G_and_S_transv_frontopol         0.190610       0.116689
    L_G_and_S_frontomargin         0.187906       0.122066
             R_G_front_sup         0.176142       0.147684
   L

## 8. Engagement outcome: views/followers ratio — Test 7

Maybe the real signal is engagement-relative-to-reach, not raw views. Predict log(views/followers) instead of log(views).

In [8]:
# Engagement = log(views / followers) — how much each video over- or under-performs the creator's reach
followers = pd.to_numeric(meta['creator_followers_at_post'], errors='coerce').fillna(0).values
views = pd.to_numeric(meta['views_at_30d'], errors='coerce').fillna(0).values
with np.errstate(divide='ignore', invalid='ignore'):
    log_engagement = np.log1p(np.where(followers > 0, views / followers, 0))
valid_eng = np.isfinite(log_engagement) & (followers > 0) & (views > 0)

print(f'Engagement outcome valid for {valid_eng.sum()}/{len(meta)} videos')
print(f'log(views/followers) distribution: median={np.median(log_engagement[valid_eng]):.2f}, std={log_engagement[valid_eng].std():.2f}')

# Top ROIs vs engagement
roi_eng_rho = np.zeros(roi_features.shape[1])
roi_eng_p = np.ones(roi_features.shape[1])
for r in range(roi_features.shape[1]):
    if roi_features[valid_eng, r].std() > 1e-9:
        roi_eng_rho[r], roi_eng_p[r] = spearmanr(roi_features[valid_eng, r], log_engagement[valid_eng])

eng_df = pd.DataFrame({
    'roi': active_roi_names,
    'rho_vs_engagement': roi_eng_rho,
    'p_vs_engagement': roi_eng_p,
}).reindex(np.abs(roi_eng_rho).argsort()[::-1]).head(10)

print(f'\nTop 10 ROIs by correlation with log(views/followers):')
print(eng_df.to_string(index=False))

n_sig_eng = (roi_eng_p < 0.01).sum()
print(f'\nROIs with p<0.01 vs engagement: {n_sig_eng}')
if n_sig_eng > 3:
    print('=> ROIs predict engagement-relative-to-reach. This is the more informative outcome.')
else:
    print('=> ROIs do NOT predict engagement. Original views correlations may be reach-driven.')

Engagement outcome valid for 69/69 videos
log(views/followers) distribution: median=2.44, std=2.47

Top 10 ROIs by correlation with log(views/followers):
                       roi  rho_vs_engagement  p_vs_engagement
R_G_and_S_transv_frontopol           0.205517         0.090246
   L_S_circular_insula_ant           0.192474         0.113091
    L_Lat_Fis-ant-Horizont           0.189660         0.118557
    L_G_and_S_frontomargin           0.188016         0.121844
    R_G_and_S_frontomargin           0.186774         0.124373
L_G_and_S_transv_frontopol           0.182499         0.133383
     L_G_front_inf-Orbital           0.181513         0.135531
       L_S_orbital_lateral           0.181220         0.136172
    L_Lat_Fis-ant-Vertical           0.177421         0.144719
   R_S_circular_insula_ant           0.159518         0.190450

ROIs with p<0.01 vs engagement: 0
=> ROIs do NOT predict engagement. Original views correlations may be reach-driven.


## 9. SUMMARY

In [9]:
print('=' * 70)
print('v4 DIAGNOSTIC SUMMARY')
print('=' * 70)

print(f'\n--- Best TRIBE feature variant ---')
print(f'{"Model":<45} {"AUC":<8} {"std":<8}  delta')
for name, (m, s) in sorted(results.items(), key=lambda kv: -kv[1][0]):
    delta = m - baseline_mean
    sign = '+' if delta >= 0 else ''
    print(f'  {name:<45} {m:.3f}    {s:.3f}    {sign}{delta:.3f}')

print(f'\n--- Bootstrap 95% CI on best delta ({best_name}) ---')
print(f'  median delta: {np.median(boot_deltas):+.3f}')
print(f'  95% CI:       [{ci_low:+.3f}, {ci_high:+.3f}]')
if ci_low > 0:
    print('  STATISTICAL VERDICT: TRIBE > baseline')
elif ci_high < 0:
    print('  STATISTICAL VERDICT: TRIBE < baseline')
else:
    print('  STATISTICAL VERDICT: indistinguishable from baseline')

print(f'\n--- Partial correlations (controlling for log_followers) ---')
print(f'  ROIs significant simple:    {(comp_df["p_simple"] < 0.01).sum()}/{len(comp_df)}')
print(f'  ROIs significant partial:   {(comp_df["p_partial"] < 0.01).sum()}/{len(comp_df)}')
print(f'  Mean rho attenuation:        {comp_df["attenuation"].mean():.3f}')

print(f'\n--- Residual analysis (TRIBE explains baseline residuals?) ---')
print(f'  Baseline R^2:               {lr_b.score(Xb_full, log_views):.3f}')
print(f'  ROIs sig vs residuals:      {n_sig_resid}/{len(roi_resid_p)} (chance: {expected_by_chance:.1f})')

print(f'\n--- Engagement outcome (views/followers) ---')
print(f'  ROIs sig vs engagement:     {n_sig_eng}/{len(roi_eng_p)}')

print(f'\n--- Bottom line ---')
best_alone = max([k for k in results if 'alone' in k], key=lambda k: results[k][0])
print(f'  Best TRIBE-alone: {best_alone} = {results[best_alone][0]:.3f}')
print(f'  Baseline alone: {baseline_mean:.3f}')
print(f'  Best combined: {best_name} = {results[best_name][0]:.3f}')

v4 DIAGNOSTIC SUMMARY

--- Best TRIBE feature variant ---
Model                                         AUC      std       delta
  Baseline                                      0.790    0.077    +0.000
  Baseline + TRIBE ROI-148 (C=0.1)              0.735    0.159    -0.054
  Baseline + TRIBE ROI-148                      0.729    0.186    -0.061
  TRIBE ROI-148 (alone)                         0.677    0.269    -0.112
  TRIBE fmri raw 20484 (alone)                  0.670    0.232    -0.119
  Baseline + TRIBE fmri PCA-30                  0.670    0.229    -0.120
  TRIBE fusion 1152 (alone)                     0.561    0.228    -0.229
  Baseline + TRIBE fusion PCA-30                0.547    0.222    -0.243

--- Bootstrap 95% CI on best delta (Baseline + TRIBE ROI-148 (C=0.1)) ---
  median delta: +0.018
  95% CI:       [-0.112, +0.150]
  STATISTICAL VERDICT: indistinguishable from baseline

--- Partial correlations (controlling for log_followers) ---
  ROIs significant simple:    12/150
  